In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [30]:
from langchain_groq import ChatGroq
model = ChatGroq(model="llama-3.3-70b-versatile",groq_api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.4'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000013C8DB73730>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000013C8E8A7F10>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [3]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi! My name is Rashmi and I am Machine Learning Engineer")])

AIMessage(content="Nice to meet you, Rashmi. It's great to hear that you're a Machine Learning Engineer. That's a fascinating field, with so many exciting developments and applications. What specific areas of machine learning are you most interested in or working on? Are you more focused on natural language processing, computer vision, or something else?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 67, 'prompt_tokens': 48, 'total_tokens': 115, 'completion_time': 0.154942508, 'completion_tokens_details': None, 'prompt_time': 0.002409717, 'prompt_tokens_details': None, 'queue_time': 0.339095837, 'total_time': 0.157352225}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ef41f-8969-7380-a23e-da93795915a2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 48, 'output_tokens': 67, 'total_to

In [4]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi! My name is Rashmi and I am Machine Learning Engineer"),
        AIMessage(content="Nice to meet you, Rashmi. It's great to hear that you're a Machine Learning Engineer. That's a fascinating field, with so many exciting developments and applications. What specific areas of machine learning are you most interested in or working on? Are you more focused on natural language processing, computer vision, or something else?"),
        HumanMessage(content="Hey! Whats my name and what do I do?")
    ]
)

AIMessage(content="Your name is Rashmi, and you're a Machine Learning Engineer.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 135, 'total_tokens': 150, 'completion_time': 0.029428649, 'completion_tokens_details': None, 'prompt_time': 0.0062005, 'prompt_tokens_details': None, 'queue_time': 0.162139949, 'total_time': 0.035629149}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ef421-fe3a-7310-9f6b-e044c8be5beb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 135, 'output_tokens': 15, 'total_tokens': 150})

### Message History
We can use Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in datastore. Future interactions will then load those messages and pass them into the chain as of the input.

In [7]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id:str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)


In [6]:
config={"configurable":{"session_id":"chat1"}}

In [9]:
response = with_message_history.invoke(
    [HumanMessage(content="Hi! My name is Rashmi and I am Machine Learning Engineer")],
    config=config
)

In [10]:
response.content

"Hello again, Rashmi! Welcome back. As a Machine Learning Engineer, you must be working on some really interesting projects. What sparked your interest in machine learning, and what do you enjoy most about your work? Is there a particular industry or problem you're passionate about solving using machine learning?"

In [11]:
with_message_history.invoke(
    [
        HumanMessage(content="What's my name?")
    ],
    config=config
)

AIMessage(content="Your name is Rashmi, and you're a Machine Learning Engineer.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 220, 'total_tokens': 235, 'completion_time': 0.020344457, 'completion_tokens_details': None, 'prompt_time': 0.031911845, 'prompt_tokens_details': None, 'queue_time': 0.325108084, 'total_time': 0.052256302}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ef42c-547b-7242-951e-bf2220b2b565-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 220, 'output_tokens': 15, 'total_tokens': 235})

In [13]:
## Change the config --> chaing session id
config1={"configurable":{"session_id":"chat2"}}
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config1
)
response

AIMessage(content="I don't know your name. I'm a large language model, I don't have the ability to recall personal information about individuals, including their names. Each time you interact with me, it's a new conversation and I don't retain any information from previous conversations. If you'd like to share your name with me, I'd be happy to chat with you!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 40, 'total_tokens': 115, 'completion_time': 0.145356343, 'completion_tokens_details': None, 'prompt_time': 0.001995032, 'prompt_tokens_details': None, 'queue_time': 0.341698903, 'total_time': 0.147351375}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ef42e-5690-7b81-9751-92cc08916b53-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 40, 'output_tokens': 75, 'tot

In [14]:
response = with_message_history.invoke(
    [HumanMessage(content="My name is Bharti?")],
    config=config1
)
response

AIMessage(content="Nice to meet you, Bharti. It's lovely to have you here. Is there something I can help you with or would you like to chat about a particular topic? I'm all ears!", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 130, 'total_tokens': 171, 'completion_time': 0.133042875, 'completion_tokens_details': None, 'prompt_time': 0.00720478, 'prompt_tokens_details': None, 'queue_time': 0.053916579, 'total_time': 0.140247655}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ef42e-e7b6-7ba1-8517-3268f702a289-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 130, 'output_tokens': 41, 'total_tokens': 171})

In [15]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config1
)
response

AIMessage(content='I remember, your name is Bharti. We just established that a moment ago.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 185, 'total_tokens': 203, 'completion_time': 0.080639586, 'completion_tokens_details': None, 'prompt_time': 0.009457423, 'prompt_tokens_details': None, 'queue_time': 0.161600205, 'total_time': 0.090097009}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ef42f-1c90-7212-a127-b19e6fcbb0b1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 185, 'output_tokens': 18, 'total_tokens': 203})

In [17]:
## Prompt Template
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant. Answer all the questions to the best of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt|model

In [18]:
chain.invoke({"messages":[HumanMessage(content="My name is Rashmi")]})

AIMessage(content='Nice to meet you, Rashmi. Is there something I can help you with or would you like to chat?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 56, 'total_tokens': 80, 'completion_time': 0.050547502, 'completion_tokens_details': None, 'prompt_time': 0.003241723, 'prompt_tokens_details': None, 'queue_time': 0.056800657, 'total_time': 0.053789225}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ef433-f12b-7d41-9f05-bfe6d1c304d0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 56, 'output_tokens': 24, 'total_tokens': 80})

In [19]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

In [20]:
config = {"configurable" : {"session_id" : "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="My name is Rashmi")],
    config=config
)

response.content

'Nice to meet you, Rashmi. Is there something I can help you with or would you like to chat?'

In [21]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assisatnt. Answer all questions to the best of your ability in {language}"
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)
chain = prompt | model

In [22]:
response = chain.invoke({"messages":[HumanMessage(content="Hi My name is Rashmi")], "language":"Hindi"})
response

AIMessage(content='नमस्ते रश्मि! मैं आपकी मदद के लिए तैयार हूँ। मुझसे क्या पूछना चाहती हैं?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 61, 'total_tokens': 104, 'completion_time': 0.129130233, 'completion_tokens_details': None, 'prompt_time': 0.005730661, 'prompt_tokens_details': None, 'queue_time': 0.162054448, 'total_time': 0.134860894}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ef457-0718-77c2-ac87-2a78d10ab328-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 61, 'output_tokens': 43, 'total_tokens': 104})

In [25]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [26]:
config = {"configurable": {"session_id": "chat4"}}
response=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi! I am Rashmi")], "language":"Hindi"},
    config=config
)
response.content

'नमस्ते रश्मि! मैं आपकी मदद के लिए तैयार हूँ। क्या आप कुछ पूछना चाहती हैं या किसी विशिष्ट विषय पर चर्चा करना चाहती हैं?'

In [27]:
response=with_message_history.invoke(
    {'messages': [HumanMessage(content="what's my name")], "language":"Hindi"},
    config=config
)
response.content

'आपका नाम रश्मि है!'

### Manging the conversation History

One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.

'trim_messgaes' -> helper to reduce how many messages we are sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages.

In [35]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer = trim_messages(
    max_tokens=70,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

messages = [
    SystemMessage(content="you are a good assistant"),
    HumanMessage(content="Hi! I am Bob"),
    AIMessage(content="Hi!"),
    HumanMessage(content="I Like Vanilla Icecream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2+2"),
    AIMessage(content="4"),
    HumanMessage(content="Thanks"),
    AIMessage(content="No Problem!"),
    HumanMessage(content="Having Fun?"),
    AIMessage(content="Yes")
]

trimmer.invoke(messages)

[SystemMessage(content='you are a good assistant', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hi! I am Bob', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Hi!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='I Like Vanilla Icecream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2+2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='No Problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Having Fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Yes', additional_kwargs={

In [37]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
)

chain.invoke(
    {
        "messages":messages + [HumanMessage(content="what math problem did I ask for")],
        "language":"English"
    }
)

AIMessage(content='You asked for the solution to the math problem: 2 + 2. The answer is 4.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 146, 'total_tokens': 169, 'completion_time': 0.088825762, 'completion_tokens_details': None, 'prompt_time': 0.050760671, 'prompt_tokens_details': None, 'queue_time': 0.494802647, 'total_time': 0.139586433}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ba38bbab80', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ef483-44d9-7370-8421-dfb7ce5d9122-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 146, 'output_tokens': 23, 'total_tokens': 169})

In [38]:
## Let's wrap this in the Message History

with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config = {"configurable":{"session_id":"chat5"}}

In [39]:
response=with_message_history.invoke(
    {'messages': [HumanMessage(content="what's my name")], "language":"Hindi"},
    config=config
)
response.content

'मुझे खेद है, लेकिन मैं आपका नाम नहीं जानता क्योंकि हमारी पहली बार बात हो रही है और आपने मुझे अपना नाम नहीं बताया है। क्या आप मुझे अपना नाम बताना चाहेंगे?'

In [40]:
response=with_message_history.invoke(
    {'messages': [HumanMessage(content="what math problem did I ask for")], "language":"English"},
    config=config
)
response.content

"You didn't ask for a math problem yet. This is the start of our conversation, and I'm ready to help with any question you might have, including math problems. What's on your mind?"